In [1]:
import os
import pandas as pd
import numpy as np

# Load the cleaned dataset from data/processed/
DATA_PATH = "../data/processed/banking_churn_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(f"Loaded processed dataset: {df.shape[0]} rows, {df.shape[1]} columns.\n")

# 1. Feature: Balance to Estimated Salary Ratio
# Avoid division by zero by setting a floor on salary
df['balance_salary_ratio'] = df['balance'] / df['estimated_salary'].replace(0, np.nan)
df['balance_salary_ratio'] = df['balance_salary_ratio'].fillna(0).round(4)

# 2. Feature: Tenure to Age Ratio (Loyalty depth)
df['tenure_age_ratio'] = (df['tenure'] / df['age']).round(4)

# 3. Feature: High Product Risk Flag (3 or 4 products)
df['is_high_product_risk'] = df['num_of_products'].apply(lambda x: 1 if x >= 3 else 0)

# 4. Feature: Zero Balance Flag (Dormant or secondary accounts)
df['is_zero_balance'] = (df['balance'] == 0).astype(int)

# Inspect newly created features
preview_cols = [
    'customer_id', 'age', 'balance', 'estimated_salary', 
    'balance_salary_ratio', 'tenure_age_ratio', 
    'num_of_products', 'is_high_product_risk', 'is_zero_balance', 'churn'
]
df[preview_cols].head()

Loaded processed dataset: 10000 rows, 14 columns.



,customer_id,age,balance,estimated_salary,balance_salary_ratio,tenure_age_ratio,num_of_products,is_high_product_risk,is_zero_balance,churn
0,15634602,42,0.00,101348.88,0.0000,0.0476,1,0,1,1
1,15647311,41,83807.86,112542.58,0.7447,0.0244,1,0,0,0
2,15619304,42,159660.80,113931.57,1.4014,0.1905,3,1,0,1
3,15701354,39,0.00,93826.63,0.0000,0.0256,2,0,1,0
4,15737888,43,125510.82,79084.10,1.5871,0.0465,1,0,0,0


In [2]:
from sklearn.model_selection import train_test_split

# 1. Define Features (X) and Target (y)
# Drop non-predictive IDs and unneeded columns
drop_cols = ['customer_id', 'surname', 'churn', 'age_group']
X = df.drop(columns=[col for col in drop_cols if col in df.columns])
y = df['churn']

# 2. One-Hot Encode categorical features (geography, gender)
X_encoded = pd.get_dummies(X, columns=['geography', 'gender'], drop_first=True, dtype=int)

# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print("=== PIPELINE DATA SPLIT SUMMARY ===")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape : {X_test.shape}  | y_test shape : {y_test.shape}")
print("-" * 45)
print(f"Train Churn Rate: {(y_train.sum() / len(y_train)) * 100:.2f}%")
print(f"Test Churn Rate : {(y_test.sum() / len(y_test)) * 100:.2f}%")
print("\nFinal Feature Matrix Columns:")
print(list(X_encoded.columns))

=== PIPELINE DATA SPLIT SUMMARY ===
X_train shape: (8000, 15) | y_train shape: (8000,)
X_test shape : (2000, 15)  | y_test shape : (2000,)
---------------------------------------------
Train Churn Rate: 20.38%
Test Churn Rate : 20.35%

Final Feature Matrix Columns:
['creditscore', 'age', 'tenure', 'balance', 'num_of_products', 'has_credit_card', 'is_active_member', 'estimated_salary', 'balance_salary_ratio', 'tenure_age_ratio', 'is_high_product_risk', 'is_zero_balance', 'geography_Germany', 'geography_Spain', 'gender_Male']


In [3]:
from sklearn.preprocessing import StandardScaler

# Identify numeric continuous columns that need scaling
continuous_cols = [
    'creditscore', 'age', 'tenure', 'balance', 
    'estimated_salary', 'balance_salary_ratio', 'tenure_age_ratio'
]

scaler = StandardScaler()

# Copy to avoid SettingWithCopy warnings
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit only on training data, transform both
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Scaling complete without data leakage!")
print(f"Sample scaled age (mean ~ 0, std ~ 1): Mean = {X_train_scaled['age'].mean():.4f}, Std = {X_train_scaled['age'].std():.4f}")

Scaling complete without data leakage!
Sample scaled age (mean ~ 0, std ~ 1): Mean = 0.0000, Std = 1.0001


In [4]:
import joblib

# Directory for preprocessed modeling splits
DATA_PROCESSED_DIR = "../data/processed"
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

# 1. Export Scaled Sets (for Linear / Distance-based models)
X_train_scaled.to_csv(f"{DATA_PROCESSED_DIR}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{DATA_PROCESSED_DIR}/X_test_scaled.csv", index=False)

# 2. Export Unscaled Sets (for Tree-based models like Random Forest / XGBoost)
X_train.to_csv(f"{DATA_PROCESSED_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{DATA_PROCESSED_DIR}/X_test.csv", index=False)

# 3. Export Target Labels
y_train.to_csv(f"{DATA_PROCESSED_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{DATA_PROCESSED_DIR}/y_test.csv", index=False)

# 4. Save the fitted scaler object for production inference
SCALER_PATH = "../data/processed/standard_scaler.joblib"
joblib.dump(scaler, SCALER_PATH)

print("Preprocessed feature splits and scaler successfully exported!")
print(f"Files saved in: {DATA_PROCESSED_DIR}")

Preprocessed feature splits and scaler successfully exported!
Files saved in: ../data/processed
